# AT3 - Avaliativo: Classificação com k-NN no Dataset Cardio

**Objetivo:** Construir um subconjunto balanceado do dataset `cardio_v2`, treinar um classificador k-NN e encontrar o melhor valor de k (de 2 a 10) com base na acurácia no conjunto de teste.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Carregamento e Inspeção do Dataset

In [ ]:
# Carrega o dataset; o separador é ponto-e-vírgula conforme o arquivo original
df = pd.read_csv('cardio_v2 (1).csv', sep=';')

print('Shape do dataset completo:', df.shape)
print('\nPrimeiras linhas:')
df.head()

In [ ]:
# Verificação da distribuição da variável alvo (cardio)
# cardio = 0: doença ausente | cardio = 1: doença presente
print('Distribuição da classe alvo (cardio):')
print(df['cardio'].value_counts())

## 2. Construção do Subconjunto Balanceado

Seleciona os primeiros 1000 registros de cada classe (0 e 1), garantindo que o dataset seja **balanceado** — condição importante para evitar viés no treinamento do k-NN. O embaralhamento ocorre na etapa seguinte, ao concatenar as classes.

In [ ]:
# Separa as duas classes
classe_0 = df[df['cardio'] == 0]
classe_1 = df[df['cardio'] == 1]

# Pega os primeiros 1000 registros de cada classe (sem aleatoriedade na seleção)
# O embaralhamento acontece logo abaixo, ao concatenar o subconjunto
SEED = 42
amostra_0 = classe_0.head(1000)
amostra_1 = classe_1.head(1000)

# Concatena e embaralha o subconjunto
subset = pd.concat([amostra_0, amostra_1]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print('Shape do subconjunto:', subset.shape)
print('\nDistribuição de classes no subconjunto:')
print(subset['cardio'].value_counts())

## 3. Pré-processamento

- Remove a coluna `id` (identificador sem valor preditivo).
- Separa features (`X`) da variável alvo (`y`).
- Normaliza as features com `StandardScaler` — o k-NN é sensível à escala das variáveis, pois usa distância euclidiana.

In [ ]:
# Remove o identificador e separa X e y
X = subset.drop(columns=['id', 'cardio'])
y = subset['cardio']

print('Features utilizadas:', list(X.columns))
print('Total de amostras:', len(X))

## 4. Divisão Treino/Teste (80/20)

In [ ]:
# Divisão estratificada para manter a proporção 50/50 em ambos os conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'Treino: {len(X_train)} amostras  |  Teste: {len(X_test)} amostras')
print('\nDistribuição no treino:')
print(y_train.value_counts())
print('\nDistribuição no teste:')
print(y_test.value_counts())

In [ ]:
# Normalização das features — FUNDAMENTAL para o k-NN
# O scaler é ajustado APENAS no treino e aplicado no teste (evita data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

## 5. Busca pelo Melhor Valor de k (k = 2 a 10)

Treina um k-NN para cada valor de k e avalia a acurácia no conjunto de teste.
O melhor k é aquele que maximiza as predições corretas.

In [ ]:
resultados = {}

for k in range(2, 11):
    # Instancia o classificador com k vizinhos e distância euclidiana (padrão)
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    
    # Treina com os dados normalizados
    knn.fit(X_train_scaled, y_train)
    
    # Prediz no conjunto de teste
    y_pred = knn.predict(X_test_scaled)
    
    # Calcula a acurácia (proporção de acertos)
    acuracia = accuracy_score(y_test, y_pred)
    acertos  = int(acuracia * len(y_test))
    
    resultados[k] = {'acuracia': acuracia, 'acertos': acertos, 'modelo': knn, 'y_pred': y_pred}
    print(f'k={k:2d} | Acurácia: {acuracia:.4f} | Acertos: {acertos}/{len(y_test)}')

# Identifica o melhor k
melhor_k = max(resultados, key=lambda k: resultados[k]['acuracia'])
print(f'\n>>> Melhor k: {melhor_k} (acurácia = {resultados[melhor_k]["acuracia"]:.4f})')

## 6. Visualização dos Resultados por k

In [ ]:
ks        = list(resultados.keys())
acuracias = [resultados[k]['acuracia'] for k in ks]

plt.figure(figsize=(8, 4))
plt.plot(ks, acuracias, marker='o', linewidth=2, color='steelblue')
plt.axvline(x=melhor_k, color='red', linestyle='--', label=f'Melhor k = {melhor_k}')
plt.xlabel('Valor de k')
plt.ylabel('Acurácia no Teste')
plt.title('Acurácia do k-NN por Valor de k')
plt.xticks(ks)
plt.legend()
plt.tight_layout()
plt.show()

## 7. Avaliação Detalhada do Melhor Modelo

In [ ]:
y_pred_melhor = resultados[melhor_k]['y_pred']

print(f'=== Relatório de Classificação (k = {melhor_k}) ===')
print(classification_report(y_test, y_pred_melhor,
                             target_names=['Sem doença (0)', 'Com doença (1)']))

In [ ]:
# Matriz de confusão do melhor modelo
cm = confusion_matrix(y_test, y_pred_melhor)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: 0', 'Pred: 1'],
            yticklabels=['Real: 0', 'Real: 1'])
plt.title(f'Matriz de Confusão — k={melhor_k}')
plt.tight_layout()
plt.show()

## 8. Conclusão

### Resultados encontrados

- O subconjunto utilizado foi **balanceado**: 1 000 amostras de cada classe, totalizando 2 000 registros.
- A divisão 80/20 resultou em **1 600 amostras de treino** e **400 de teste**, mantendo o equilíbrio entre classes em ambos os conjuntos.
- A normalização com `StandardScaler` foi essencial, pois features como `age` (em dias) e `weight` (em kg) possuem escalas muito diferentes; sem normalização, o k-NN tenderia a ignorar features de menor magnitude.
- Testando k de 2 a 10, o **melhor valor de k foi identificado automaticamente** como aquele que maximizou a acurácia no conjunto de teste.
- Em geral, valores muito baixos de k (como k=2) tendem a **overfitting** (muito sensíveis a ruído), enquanto valores muito altos podem **suavizar demais** a fronteira de decisão. O melhor k representa um equilíbrio entre esses extremos.
- A acurácia obtida demonstra que o k-NN consegue separar com razoável precisão pacientes com e sem doença cardiovascular a partir das features disponíveis no dataset.

### Observação sobre o dataset
O dataset original contém algumas entradas com valores fisiologicamente implausíveis (ex.: pressão arterial negativa, como `ap_lo = -1`). Em um cenário de produção, essas inconsistências deveriam ser tratadas; para fins deste exercício, foram mantidas conforme o dataset fornecido.